# La Liga Web Scraping Pipeline
## Project Overview
Before we can predict who wins a football match, we need high-quality, granular data. This project implements an end-to-end web scraping pipeline to extract multi-season match results and detailed shooting statistics for La Liga from FBref.

Rather than just downloading a static dataset, we will be web scraping the data. In this project we will handle recursive page navigation (scraping historical seasons dynamically), merge seperate data sources, manage request throttling to respect host servers, and handles missing data gracefully using exception handling.

The final output being a clean, feature-rich dataset of 1,400+ historical matches saved directly to a CSV, fully optimized for downstream machine learning tasks.

### Scraping our First Page with Requests

In [ ]:
import ssl
# Force Python to ignore unverified SSL certificates
ssl._create_default_https_context = ssl._create_unverified_context

import undetected_chromedriver as uc
from bs4 import BeautifulSoup
import time

standings_url = "https://fbref.com/en/comps/12/La-Liga-Stats"

print("Starting browser...")
options = uc.ChromeOptions()
driver = uc.Chrome(options=options)

print("Navigating to FBref...")
driver.get(standings_url)

# Give Cloudflare a few seconds to run its JavaScript challenge
time.sleep(5) 

# Grab the fully rendered HTML after the challenge is passed
html = driver.page_source

# Close the browser
driver.quit()

Starting browser...
Navigating to FBref...
Success! Found these squad links:
['/en/squads/206d90db/Barcelona-Stats', '/en/squads/53a2f082/Real-Madrid-Stats', '/en/squads/2a8183b3/Villarreal-Stats', '/en/squads/db3b9613/Atletico-Madrid-Stats', '/en/squads/fc536746/Real-Betis-Stats', '/en/squads/f25da7fb/Celta-Vigo-Stats', '/en/squads/7848bd64/Getafe-Stats', '/en/squads/98e8af82/Rayo-Vallecano-Stats', '/en/squads/dcc91a7b/Valencia-Stats', '/en/squads/e31d1cd9/Real-Sociedad-Stats', '/en/squads/a8661628/Espanyol-Stats', '/en/squads/2b390eca/Athletic-Club-Stats', '/en/squads/ad2be733/Sevilla-Stats', '/en/squads/8d6fd021/Alaves-Stats', '/en/squads/6c8b07df/Elche-Stats', '/en/squads/9800b6a1/Levante-Stats', '/en/squads/03c57e2b/Osasuna-Stats', '/en/squads/2aa12281/Mallorca-Stats', '/en/squads/9024a00a/Girona-Stats', '/en/squads/ab358912/Oviedo-Stats']


### Parsing the HTML Links with BeautifulSoup
In order to parse the HTML we are going to use the BeautifulSoup Library, which is just a popular Python library used for parsing HTML and XML documents to extract, navigate, and modify web data. We will be scraping the data the following way: 

1. **League Standings Parser**:
<br>Download the La Liga standings table, parse the HTML DOM, and use CSS selectors to isolate the specific anchor (< a >) tags belonging to individual squad pages.
2. **Form & Fixture Extractor**:
<br>Navigates to each squad's unique URL and use Pandas' read_html function to convert the Scores & Fixtures table into a DataFrame.
3. **Granular Shooting Stats Scraper**:
<br>Locates the relative link to the team's detailed shooting page, download it, drop multi-level header indexes, and extracts advanced metrics (`Shots`, `Shots on Target`, `Shot Distance`, `FK`, `PK`).


In [ ]:
# Hand the HTML off to BeautifulSoup
soup = BeautifulSoup(html, features="html.parser")

# Safely extract the table
tables = soup.select('table.stats_table')

if len(tables) == 0:
    print("Error: Website Blocked Web Scraping")
else:
    # Getting the table we need from the HTML 
    standings_table = tables[0]
    # Getting the anchor tags belonging to individual squad pages.
    links = standings_table.find_all('a')
    # Getting the href elements from the anchor tags
    href_links = [l.get("href") for l in links]
    # Filtering our any links that are not part of the squad list 
    squad_links = [l for l in href_links if l and '/squads/' in l]
    
    # Sanity Check
    squad_links